In [ ]:
# # Import pandas and numpy libraries for data analysis and processing 
import pandas as pd
import numpy as np

In [ ]:
# Read the CSV dataset and display the first 5 rows to inspect the initial data structure 
data = pd.read_csv("dataset.csv")
data.head()

,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [ ]:
# Check the overall dimensions (rows and columns) of the dataset
data.shape

(114000, 21)

In [ ]:
# Check structural information of the dataset including data types and non-null counts
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 114000 entries, 0 to 113999
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Unnamed: 0        114000 non-null  int64  
 1   track_id          114000 non-null  str    
 2   artists           113999 non-null  str    
 3   album_name        113999 non-null  str    
 4   track_name        113999 non-null  str    
 5   popularity        114000 non-null  int64  
 6   duration_ms       114000 non-null  int64  
 7   explicit          114000 non-null  bool   
 8   danceability      114000 non-null  float64
 9   energy            114000 non-null  float64
 10  key               114000 non-null  int64  
 11  loudness          114000 non-null  float64
 12  mode              114000 non-null  int64  
 13  speechiness       114000 non-null  float64
 14  acousticness      114000 non-null  float64
 15  instrumentalness  114000 non-null  float64
 16  liveness          114000 non-nu

In [ ]:
# Extract and display the full list of column names in the dataset
data.columns.to_list()

['Unnamed: 0',
 'track_id',
 'artists',
 'album_name',
 'track_name',
 'popularity',
 'duration_ms',
 'explicit',
 'danceability',
 'energy',
 'key',
 'loudness',
 'mode',
 'speechiness',
 'acousticness',
 'instrumentalness',
 'liveness',
 'valence',
 'tempo',
 'time_signature',
 'track_genre']

In [ ]:
# Calculate the total number of missing values across the entire dataset
data.isna().sum().sum()

np.int64(3)

In [ ]:
# Define a custom function to comprehensively audit the dataset
# (including dimensions, duplicate detection, missing values, and distinct genres)
def audit_dataset(data):
    print("BASIC SHAPE")
    print(f"Rows: {len(data):,} | Columns: {data.shape[1]}")

    print("\n DUPLICATES")
    print(f"Full-row duplicates: {data.duplicated().sum():,}")
    print(f"Duplicates by track_id: {data.duplicated(subset=['track_id']).sum():,}")
    print(f"Duplicates by (name, artist): {data.duplicated(subset=['track_name','artists']).sum():,}")

    print("\n MISSING VALUES (top 10)")
    print(data.isna().sum().sort_values(ascending=False).head(10))

    print("\nGENRE COUNT")
    print(f"Distinct genres: {data['track_genre'].nunique()}")

audit_dataset(data)

BASIC SHAPE
Rows: 114,000 | Columns: 21

 DUPLICATES
Full-row duplicates: 0
Duplicates by track_id: 24,259
Duplicates by (name, artist): 32,656

 MISSING VALUES (top 10)
artists         1
track_name      1
album_name      1
Unnamed: 0      0
track_id        0
popularity      0
duration_ms     0
explicit        0
danceability    0
energy          0
dtype: int64

GENRE COUNT
Distinct genres: 114


In [8]:
# Unnamed: 0 is a unique id for each song which we don't need
data.drop("Unnamed: 0", inplace=True, axis=1)

In [ ]:
# Identify and store text column names to perform cleaning operations on them
text_cols = data.select_dtypes(include=["object", "string"]).columns.to_list()
text_cols

['track_id', 'artists', 'album_name', 'track_name', 'track_genre']

In [ ]:
# Define values that unintentionally act as text but represent missing values (NaN)
fake_nulls = ["", " ", "none", "null", "nan", "n/a", "na", "unknown", "?", "-"]

# Clean text: strip whitespace, convert to lowercase, and replace invalid values with NaN
for col in text_cols:
    data[col] = data[col].str.strip()
    data[col] = data[col].str.lower()
    data[col] = data[col].replace(fake_nulls, np.nan)

In [ ]:
# Recalculate total missing values after replacing invalid text values with NaN
data.isna().sum().sum()

np.int64(24)

In [ ]:
# Drop all rows that contain at least one missing value (NaN)
data.dropna(inplace=True)

In [ ]:
# Check and print the number of duplicate rows based on full row, track_id, and track name/artist combinationprint("DUPLICATES:\n")
print(f"with no subset: {data.duplicated().sum()}")
print(f"track_id as subset: {data.duplicated(subset="track_id").sum():,}")
print(f"track_name and artists as subset: {data.duplicated(subset=["track_name", "artists"]).sum():,}")

DUPLICATES:

with no subset: 450
track_id as subset: 24,255
track_name and artists as subset: 32,783


In [ ]:
# Drop the 'track_id' column as it is no longer needed for subsequent analysis
data.drop("track_id", inplace=True, axis=1)

In [ ]:
print("DUPLICATES BASED ON AUDIO FEATURES")
audio_cols = [
    "duration_ms", "danceability", "energy", "loudness",
    "speechiness", "acousticness", "instrumentalness",
    "liveness", "valence", "tempo",
]

# Build a rounded fingerprint so tiny float noise doesn't split identical tracks
data["_fp"] = data[audio_cols].round(3).astype(str).agg("|".join, axis=1)

# Keep the copy with the highest popularity (best proxy for "canonical")
before = len(data)
data = (
    data.sort_values(["_fp", "popularity"], ascending=[True, False])
        .drop_duplicates(subset="_fp", keep="first")
        .drop(columns="_fp")
        .reset_index(drop=True)
)
print(f"dropped {before - len(data):,} same-recording duplicates")

DUPLICATES BASED ON AUDIO FEATURES
dropped 30,522 same-recording duplicates


In [ ]:
# Extract integer columns
int_cols = data.select_dtypes(include=["int64", "Int64"]).columns.tolist()
print("integer columns:", int_cols)

# Define valid ranges for each integer feature
valid_ranges = {
    "popularity": (0, 100),
    "duration_ms": (1, np.inf),
    "key": (-1, 11),
    "mode": (0, 1),
    "time_signature": (3, 7),
}

# Check values, convert out-of-range values to NaN, and fill with the median
for col, (low, high) in valid_ranges.items():
   if col in data.columns:
      data[col] = pd.to_numeric(data[col], errors="coerce")
      data.loc[~data[col].between(low, high), col] = np.nan
      data[col] = data[col].fillna(data[col].median())

integer columns: ['popularity', 'duration_ms', 'key', 'mode', 'time_signature']


In [ ]:
# Extract float columns (audio features)
float_cols = data.select_dtypes(include=["float64", "Float64"]).columns.to_list()

# Define standard valid ranges for audio features
valid_ranges2 = {
    "danceability": (0.0, 1.0),
    "energy": (0.0, 1.0),
    "speechiness": (0.0, 1.0),
    "acousticness": (0.0, 1.0),
    "instrumentalness": (0.0, 1.0),
    "liveness": (0.0, 1.0),
    "valence": (0.0, 1.0),
    "loudness": (-60.0, 0.0),
    "tempo": (1.0, 250.0),
}

# Validate ranges, handle out-of-range values, and fill with column median
for col, (low, high) in valid_ranges2.items():
   if col in data.columns:
      data[col] = pd.to_numeric(data[col], errors="coerce")
      data.loc[~data[col].between(low, high), col] = np.nan
      data[col] = data[col].fillna(data[col].median())

In [ ]:
# Convert explicit column data type from boolean (True/False) to integer (1/0)
data.explicit = data.explicit.astype(int) # change bool to int

In [ ]:
# Convert track duration from milliseconds to minutes, round to 1 decimal place, 
# and rename column from duration_ms to duration_m
data["duration_ms"] = round(data["duration_ms"] / 60000, 1)
data.rename(columns={"duration_ms": "duration_m"}, inplace=True)

In [ ]:
# Display the final rows of the dataset to verify the cleaning and preprocessing results
data.head()

,artists,album_name,track_name,popularity,duration_m,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,deuter,space,warmer sommerabend,23.0,16.7,0,0.203,0.025,3.0,-29.156,0.0,0.0447,0.9850,0.769000,0.1020,0.0361,134.146,4.0,new-age
1,los hermanos campos,30 cuecas,amigo mío,42.0,1.7,0,0.582,0.876,1.0,-5.615,0.0,0.0595,0.8320,0.000023,0.3970,0.8200,124.890,3.0,folk
2,beat oven,beat soup,mom's spaghetti,46.0,1.7,0,0.695,0.323,4.0,-9.748,0.0,0.1550,0.0029,0.685000,0.0747,0.2950,179.861,4.0,study
3,luis pescetti;marta gómez,magia todo el día,sos mi sol flaquito,39.0,1.7,0,0.787,0.223,0.0,-12.000,1.0,0.0601,0.8480,0.000052,0.0885,0.7780,109.322,4.0,latino
4,wavcrush,"well, come",the castle,5.0,1.7,0,0.787,0.318,3.0,-9.155,0.0,0.0736,0.5420,0.930000,0.0996,0.3530,78.010,4.0,study
